# MedGuard AI: Evidence-Constrained Clinical Prescription Review Copilot

This notebook builds a hackathon-ready clinical review agent using:

- **ROCm + vLLM** for serving Qwen3 on AMD GPUs
- **Qwen3** for evidence-grounded reasoning and intervention drafting
- **PydanticAI** for typed agent orchestration and tool calling
- **FHIR-style synthetic patient data** for a US healthcare workflow

The goal is to review a proposed prescription against medication history, medical history, allergies, labs, drug interactions, DUR rules, and controlled-substance risk.

This tutorial includes the following sections:

- [1. Launching the vLLM server](#step1)
- [2. Installing dependencies](#step2)
- [3. Connecting PydanticAI to Qwen3](#step3)
- [4. Defining clinical review schemas](#step4)
- [5. Creating synthetic FHIR-style patient data](#step5)
- [6. Building DUR and evidence tools](#step6)
- [7. Creating the MedGuard AI agent](#step7)
- [8. Running a clinical prescription review](#step8)
- [9. Evaluating the solution](#step9)
- [10. Challenge ideas for the hackathon](#step10)

> Safety note: this is a clinical decision-support prototype for synthetic data. It must not be used to make autonomous medication decisions.

<a id="step1"></a>

## Step 1: Launching the vLLM server

Start a vLLM OpenAI-compatible endpoint for Qwen3 on an AMD GPU with ROCm. Open a terminal in your Jupyter environment and run one of the commands below.

For a strong hackathon demo, use `Qwen/Qwen3-30B-A3B` if your GPU memory allows it. For a lighter setup, use `Qwen/Qwen3-14B` or `Qwen/Qwen3-8B`.

> **Copy and paste this command to launch your vLLM server:**

```bash
vllm serve Qwen/Qwen3-30B-A3B \
  --host 0.0.0.0 \
  --port 8000 \
  --served-model-name Qwen3-30B-A3B \
  --trust-remote-code \
  --max-model-len 32768 \
  --gpu-memory-utilization 0.90
```

If you need a smaller model:

```bash
vllm serve Qwen/Qwen3-14B \
  --host 0.0.0.0 \
  --port 8000 \
  --served-model-name Qwen3-14B \
  --trust-remote-code
```

In [ ]:
import os

BASE_URL = "http://localhost:8000/v1"
MODEL_NAME = os.getenv("MODEL_NAME", "Qwen3-30B-A3B")

os.environ["BASE_URL"] = BASE_URL
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "abc-123")

print("Config set")
print("BASE_URL:", BASE_URL)
print("MODEL_NAME:", MODEL_NAME)

Verify the model is available. If this cell fails, keep going: the deterministic clinical-review engine later in the notebook works without a running LLM server.

In [ ]:
!curl http://localhost:8000/v1/models -H "Authorization: Bearer $OPENAI_API_KEY"

<a id="step2"></a>

## Step 2: Installing dependencies

The notebook uses PydanticAI for the agent, Pydantic for typed outputs, and Rich for readable display.

In [ ]:
!pip install -q pydantic pydantic_ai openai rich

<a id="step3"></a>

## Step 3: Connecting PydanticAI to Qwen3

This mirrors the sample notebook pattern: create an OpenAI-compatible provider pointed at the vLLM endpoint, then create a PydanticAI agent model.

In [ ]:
try:
    from pydantic_ai.models.openai import OpenAIModel
    from pydantic_ai.providers.openai import OpenAIProvider

    provider = OpenAIProvider(
        base_url=os.environ["BASE_URL"],
        api_key=os.environ["OPENAI_API_KEY"],
    )

    agent_model = OpenAIModel(os.environ.get("MODEL_NAME", MODEL_NAME), provider=provider)
    print("PydanticAI model configured:", MODEL_NAME)
except Exception as exc:
    agent_model = None
    print("PydanticAI model was not configured yet.")
    print("Run the dependency install cell and make sure your vLLM server is available.")
    print(type(exc).__name__, str(exc)[:300])

In [ ]:
import asyncio

async def run_async(prompt: str):
    if agent is None:
        raise RuntimeError("MedGuard agent is not configured. Install pydantic_ai and configure the vLLM endpoint.")
    result = await agent.run(prompt)
    return result.output

<a id="step4"></a>

## Step 4: Defining clinical review schemas

Clinical review needs structure. The agent will return a typed decision with findings, citations, and intervention text.

In [ ]:
from typing import Literal, Optional
from pydantic import BaseModel, Field


Severity = Literal["low", "moderate", "high", "critical"]
Action = Literal["approve", "monitor", "clarify", "substitute", "block"]


class ProposedPrescription(BaseModel):
    drug_name: str
    dose: str
    route: str
    frequency: str
    duration: str
    indication: str


class EvidenceCitation(BaseModel):
    source: str
    title: str
    url: str
    note: str


class Finding(BaseModel):
    category: Literal[
        "drug_interaction",
        "allergy",
        "contraindication",
        "duplicate_therapy",
        "dose_adjustment",
        "monitoring",
        "controlled_substance",
        "missing_data",
    ]
    severity: Severity
    title: str
    rationale: str
    recommendation: str
    evidence_ids: list[str] = Field(default_factory=list)


class ClinicalReviewDecision(BaseModel):
    patient_id: str
    prescription: ProposedPrescription
    overall_risk: Severity
    recommended_action: Action
    key_findings: list[Finding]
    missing_information: list[str] = Field(default_factory=list)
    evidence: list[EvidenceCitation] = Field(default_factory=list)
    clinician_message: str
    pharmacist_note: str

<a id="step5"></a>

## Step 5: Creating synthetic FHIR-style patient data

For a hackathon, avoid real PHI. Use synthetic FHIR-like records generated by Synthea or created manually as below.

This case is intentionally high signal:

- Older adult with atrial fibrillation on warfarin
- CKD stage 3b with elevated potassium
- Documented sulfonamide allergy
- Existing opioid and benzodiazepine overlap
- Proposed prescription: TMP-SMX for UTI

In [ ]:
from pprint import pprint

PATIENTS = {
    "patient-001": {
        "resourceType": "Bundle",
        "patient": {
            "id": "patient-001",
            "name": "Synthetic Patient",
            "age": 74,
            "sex": "female",
        },
        "conditions": [
            "Atrial fibrillation",
            "Chronic kidney disease stage 3b",
            "Type 2 diabetes mellitus",
            "Hypertension",
            "Obstructive sleep apnea",
        ],
        "allergies": [
            {
                "substance": "sulfonamide antibiotics",
                "reaction": "anaphylaxis",
                "severity": "severe",
            },
            {
                "substance": "penicillin",
                "reaction": "rash",
                "severity": "moderate",
            },
        ],
        "medications": [
            {"name": "warfarin", "dose": "5 mg", "route": "oral", "frequency": "daily"},
            {"name": "lisinopril", "dose": "20 mg", "route": "oral", "frequency": "daily"},
            {"name": "spironolactone", "dose": "25 mg", "route": "oral", "frequency": "daily"},
            {"name": "metformin", "dose": "1000 mg", "route": "oral", "frequency": "twice daily"},
            {"name": "alprazolam", "dose": "0.5 mg", "route": "oral", "frequency": "as needed"},
            {"name": "oxycodone", "dose": "5 mg", "route": "oral", "frequency": "every 6 hours as needed"},
        ],
        "observations": {
            "eGFR": {"value": 38, "unit": "mL/min/1.73m2"},
            "potassium": {"value": 5.3, "unit": "mmol/L"},
            "INR": {"value": 3.2, "unit": "ratio"},
            "AST": {"value": 32, "unit": "U/L"},
            "ALT": {"value": 29, "unit": "U/L"},
        },
    }
}

proposed_rx = ProposedPrescription(
    drug_name="sulfamethoxazole-trimethoprim DS",
    dose="800 mg / 160 mg",
    route="oral",
    frequency="twice daily",
    duration="7 days",
    indication="uncomplicated urinary tract infection",
)

pprint(PATIENTS["patient-001"])
proposed_rx

<a id="step6"></a>

## Step 6: Building DUR and evidence tools

The key design principle is: deterministic tools identify safety facts, and the LLM explains them with citations. This reduces hallucination risk and makes the system more enterprise-ready.

In [ ]:
DRUG_DB = {
    "warfarin": {
        "ingredients": ["warfarin"],
        "classes": ["anticoagulant", "vitamin_k_antagonist"],
        "controlled": False,
    },
    "sulfamethoxazole-trimethoprim ds": {
        "ingredients": ["sulfamethoxazole", "trimethoprim"],
        "classes": ["sulfonamide_antibiotic", "folate_antagonist_antibiotic"],
        "controlled": False,
    },
    "tmp-smx": {
        "ingredients": ["sulfamethoxazole", "trimethoprim"],
        "classes": ["sulfonamide_antibiotic", "folate_antagonist_antibiotic"],
        "controlled": False,
    },
    "lisinopril": {
        "ingredients": ["lisinopril"],
        "classes": ["ace_inhibitor"],
        "controlled": False,
    },
    "spironolactone": {
        "ingredients": ["spironolactone"],
        "classes": ["potassium_sparing_diuretic", "aldosterone_antagonist"],
        "controlled": False,
    },
    "metformin": {
        "ingredients": ["metformin"],
        "classes": ["biguanide"],
        "controlled": False,
    },
    "alprazolam": {
        "ingredients": ["alprazolam"],
        "classes": ["benzodiazepine"],
        "controlled": True,
        "schedule": "C-IV",
    },
    "oxycodone": {
        "ingredients": ["oxycodone"],
        "classes": ["opioid_analgesic"],
        "controlled": True,
        "schedule": "C-II",
        "mme_factor": 1.5,
    },
}


EVIDENCE_KB = {
    "dailyMed_tmp_smx": {
        "source": "DailyMed",
        "title": "Sulfamethoxazole and Trimethoprim labeling",
        "url": "https://dailymed.nlm.nih.gov/dailymed/",
        "note": "FDA labeling source for indications, warnings, contraindications, and adverse reactions.",
    },
    "dailyMed_warfarin": {
        "source": "DailyMed",
        "title": "Warfarin labeling",
        "url": "https://dailymed.nlm.nih.gov/dailymed/",
        "note": "FDA labeling source for warfarin interaction and bleeding-risk warnings.",
    },
    "rxnorm": {
        "source": "NLM RxNorm/RxNav",
        "title": "RxNorm and RxNav APIs",
        "url": "https://lhncbc.nlm.nih.gov/RxNav/APIs/index.html",
        "note": "Medication normalization and drug concept mapping.",
    },
    "cdc_opioid": {
        "source": "CDC",
        "title": "CDC opioid prescribing clinical guidance",
        "url": "https://www.cdc.gov/overdose-prevention/hcp/clinical-guidance/index.html",
        "note": "Guidance for opioid risk review; does not replace clinical judgment.",
    },
    "fhir_medication_request": {
        "source": "HL7 FHIR",
        "title": "FHIR MedicationRequest",
        "url": "https://hl7.org/fhir/R4/medicationrequest.html",
        "note": "FHIR resource for medication orders or requests.",
    },
}


INTERACTION_RULES = [
    {
        "pair": {"warfarin", "sulfamethoxazole"},
        "severity": "critical",
        "title": "TMP-SMX may increase warfarin anticoagulation effect",
        "rationale": "Patient is already anticoagulated and has an INR of 3.2. TMP-SMX is a high-risk interacting antibiotic in a warfarin patient.",
        "recommendation": "Block pending clinician/pharmacist review. Consider an alternative antibiotic or define an INR monitoring and dose-adjustment plan.",
        "evidence_ids": ["dailyMed_tmp_smx", "dailyMed_warfarin"],
    },
    {
        "pair": {"lisinopril", "spironolactone"},
        "severity": "high",
        "title": "ACE inhibitor plus potassium-sparing diuretic with elevated potassium",
        "rationale": "Patient takes lisinopril and spironolactone and has potassium 5.3 mmol/L.",
        "recommendation": "Flag baseline hyperkalemia risk and confirm potassium monitoring before adding medications that may worsen kidney or potassium status.",
        "evidence_ids": ["dailyMed_tmp_smx"],
    },
    {
        "pair": {"oxycodone", "alprazolam"},
        "severity": "high",
        "title": "Opioid and benzodiazepine overlap",
        "rationale": "Patient has active oxycodone and alprazolam prescriptions and obstructive sleep apnea.",
        "recommendation": "Review controlled-substance risk, sedation risk, naloxone need, and PDMP history if available.",
        "evidence_ids": ["cdc_opioid"],
    },
]

In [ ]:
def _canon(name: str) -> str:
    return name.lower().replace(",", " ").replace("  ", " ").strip()


def normalize_medication_name(name: str) -> dict:
    """Normalize a medication name to ingredients and classes using a local RxNorm-like map."""
    key = _canon(name)
    if key in DRUG_DB:
        return {"input": name, "normalized_name": key, **DRUG_DB[key]}

    if "sulfamethoxazole" in key or "trimethoprim" in key or "bactrim" in key:
        return {"input": name, "normalized_name": "sulfamethoxazole-trimethoprim ds", **DRUG_DB["sulfamethoxazole-trimethoprim ds"]}

    return {
        "input": name,
        "normalized_name": key,
        "ingredients": [key],
        "classes": [],
        "controlled": False,
    }


def patient_ingredients(patient: dict) -> list[str]:
    ingredients = []
    for med in patient["medications"]:
        normalized = normalize_medication_name(med["name"])
        ingredients.extend(normalized["ingredients"])
    return sorted(set(ingredients))


def patient_classes(patient: dict) -> list[str]:
    classes = []
    for med in patient["medications"]:
        normalized = normalize_medication_name(med["name"])
        classes.extend(normalized["classes"])
    return sorted(set(classes))


normalize_medication_name(proposed_rx.drug_name)

In [ ]:
try:
    from pydantic_ai import Tool
except Exception:
    def Tool(func=None):
        if func is None:
            return lambda f: f
        return func

    print("PydanticAI is not available yet. Using no-op Tool decorator for offline baseline.")


@Tool
def get_patient_context(patient_id: str) -> dict:
    """Return the synthetic FHIR-style patient context for a patient id."""
    return PATIENTS[patient_id]


@Tool
def normalize_medication(name: str) -> dict:
    """Normalize a medication name to ingredients, classes, and controlled-substance attributes."""
    return normalize_medication_name(name)


@Tool
def retrieve_evidence(evidence_ids: list[str]) -> list[dict]:
    """Return evidence citations for the requested evidence ids."""
    return [EVIDENCE_KB[eid] for eid in evidence_ids if eid in EVIDENCE_KB]

In [ ]:
def _finding_from_rule(rule: dict) -> Finding:
    return Finding(
        category="drug_interaction",
        severity=rule["severity"],
        title=rule["title"],
        rationale=rule["rationale"],
        recommendation=rule["recommendation"],
        evidence_ids=rule["evidence_ids"],
    )


def _severity_rank(severity: str) -> int:
    return {"low": 1, "moderate": 2, "high": 3, "critical": 4}[severity]


def _overall_risk(findings: list[Finding]) -> Severity:
    if not findings:
        return "low"
    return max((f.severity for f in findings), key=_severity_rank)


def _recommended_action(overall_risk: Severity) -> Action:
    return {
        "low": "approve",
        "moderate": "monitor",
        "high": "clarify",
        "critical": "block",
    }[overall_risk]


def run_deterministic_dur_review(patient_id: str, rx: ProposedPrescription) -> dict:
    patient = PATIENTS[patient_id]
    proposed = normalize_medication_name(rx.drug_name)
    active_ingredients = set(patient_ingredients(patient))
    proposed_ingredients = set(proposed["ingredients"])
    all_ingredients = active_ingredients | proposed_ingredients
    active_classes = set(patient_classes(patient))
    proposed_classes = set(proposed["classes"])

    findings: list[Finding] = []

    # Allergy review.
    for allergy in patient["allergies"]:
        allergy_text = allergy["substance"].lower()
        if "sulfonamide" in allergy_text and "sulfonamide_antibiotic" in proposed_classes:
            findings.append(
                Finding(
                    category="allergy",
                    severity="critical" if allergy["severity"] == "severe" else "high",
                    title="Sulfonamide antibiotic allergy conflict",
                    rationale=(
                        f"Patient has {allergy['severity']} allergy to {allergy['substance']} "
                        f"with reaction '{allergy['reaction']}', and the proposed medication contains sulfamethoxazole."
                    ),
                    recommendation="Block pending clinician/pharmacist review and clarify allergy history or select a non-sulfonamide alternative.",
                    evidence_ids=["dailyMed_tmp_smx"],
                )
            )

    # Drug-drug and background-risk review.
    for rule in INTERACTION_RULES:
        if rule["pair"].issubset(all_ingredients):
            findings.append(_finding_from_rule(rule))

    # Renal and lab monitoring review.
    egfr = patient["observations"].get("eGFR", {}).get("value")
    potassium = patient["observations"].get("potassium", {}).get("value")
    inr = patient["observations"].get("INR", {}).get("value")

    if egfr is None:
        findings.append(
            Finding(
                category="missing_data",
                severity="high",
                title="Missing kidney function",
                rationale="Renal function is required before reviewing dosing and monitoring for many antimicrobials.",
                recommendation="Obtain recent serum creatinine/eGFR before dispensing.",
                evidence_ids=[],
            )
        )
    elif egfr < 45 and proposed_ingredients.intersection({"sulfamethoxazole", "trimethoprim"}):
        findings.append(
            Finding(
                category="dose_adjustment",
                severity="moderate",
                title="Reduced kidney function requires dosing and monitoring review",
                rationale=f"Patient eGFR is {egfr} mL/min/1.73m2. TMP-SMX warrants renal function and potassium monitoring.",
                recommendation="Confirm renal dosing policy and monitoring plan before use.",
                evidence_ids=["dailyMed_tmp_smx"],
            )
        )

    if potassium is not None and potassium >= 5.0 and proposed_ingredients.intersection({"trimethoprim"}):
        findings.append(
            Finding(
                category="monitoring",
                severity="high",
                title="Elevated potassium before trimethoprim exposure",
                rationale=f"Patient potassium is {potassium} mmol/L and already has CKD plus ACE inhibitor/spironolactone exposure.",
                recommendation="Clarify whether TMP-SMX is necessary; consider alternative and repeat potassium monitoring.",
                evidence_ids=["dailyMed_tmp_smx"],
            )
        )

    if inr is not None and inr > 3.0 and proposed_ingredients.intersection({"sulfamethoxazole", "trimethoprim"}):
        findings.append(
            Finding(
                category="monitoring",
                severity="critical",
                title="Supratherapeutic INR before interacting antibiotic",
                rationale=f"Patient INR is {inr}. Adding TMP-SMX to warfarin may increase bleeding risk.",
                recommendation="Block pending anticoagulation plan, alternative antibiotic, or near-term INR monitoring.",
                evidence_ids=["dailyMed_tmp_smx", "dailyMed_warfarin"],
            )
        )

    evidence_ids = sorted({eid for finding in findings for eid in finding.evidence_ids})
    evidence = [EvidenceCitation(**EVIDENCE_KB[eid]) for eid in evidence_ids if eid in EVIDENCE_KB]
    overall = _overall_risk(findings)
    action = _recommended_action(overall)

    return ClinicalReviewDecision(
        patient_id=patient_id,
        prescription=rx,
        overall_risk=overall,
        recommended_action=action,
        key_findings=sorted(findings, key=lambda f: _severity_rank(f.severity), reverse=True),
        missing_information=[],
        evidence=evidence,
        clinician_message=(
            "High-risk prescription review: proposed TMP-SMX conflicts with documented sulfonamide allergy, "
            "active warfarin therapy with INR 3.2, CKD, and elevated potassium. Recommend blocking pending review "
            "and considering an alternative antibiotic or a documented monitoring plan."
        ),
        pharmacist_note=(
            "DUR: Do not dispense until prescriber clarification. Documented severe sulfonamide allergy; "
            "warfarin interaction with INR 3.2; CKD stage 3b with K 5.3 on lisinopril/spironolactone. "
            "Recommend alternative therapy or explicit INR/potassium monitoring plan."
        ),
    ).model_dump()


@Tool
def run_dur_review(patient_id: str, prescription: dict) -> dict:
    """Run deterministic DUR, allergy, interaction, renal, lab, and controlled-substance screening."""
    rx = ProposedPrescription(**prescription)
    return run_deterministic_dur_review(patient_id, rx)

Run the clinical review engine before adding the LLM. This gives you a deterministic baseline that can be evaluated and audited.

In [ ]:
try:
    from rich import print as rprint
except Exception:
    rprint = print

baseline_review = run_deterministic_dur_review("patient-001", proposed_rx)
rprint(baseline_review)

<a id="step7"></a>

## Step 7: Creating the MedGuard AI agent

The agent should not invent clinical facts. Its job is to call tools, preserve structured findings, add concise reasoning, and produce a pharmacist-ready intervention note.

In [ ]:
try:
    from pydantic_ai import Agent
except Exception as exc:
    Agent = None
    print("PydanticAI Agent is not available yet.")
    print(type(exc).__name__, str(exc)[:300])

system_prompt = """
You are MedGuard AI, a clinical prescription review copilot for synthetic US healthcare data.

Rules:
- You are clinical decision support, not an autonomous prescriber.
- Always call get_patient_context and run_dur_review before giving a recommendation.
- Preserve deterministic findings and severity from the DUR tool.
- Use retrieve_evidence for citation ids returned by findings.
- If data is missing, say what is missing.
- Keep the clinician message concise and actionable.
- Do not claim that FAERS or spontaneous adverse event reports prove causality.
"""

if Agent is None or agent_model is None:
    agent = None
    print("Agent skipped. The deterministic review engine remains available.")
else:
    agent_kwargs = dict(
        model=agent_model,
        tools=[
            get_patient_context,
            normalize_medication,
            retrieve_evidence,
            run_dur_review,
        ],
        system_prompt=system_prompt,
    )

    try:
        agent = Agent(**agent_kwargs, output_type=ClinicalReviewDecision)
        print("Agent created with typed output.")
    except TypeError:
        agent = Agent(**agent_kwargs)
        print("Agent created without typed-output constructor support. Prompt will request JSON output.")

<a id="step8"></a>

## Step 8: Running a clinical prescription review

If your vLLM server is running, this cell asks Qwen3 to orchestrate the tools and produce the final clinical decision. If the LLM endpoint is unavailable, use the deterministic `baseline_review` from the previous section for the demo.

In [ ]:
review_prompt = f"""
Review this proposed prescription for patient-001.

Proposed prescription:
{proposed_rx.model_dump()}

Return a structured clinical review with:
- overall_risk
- recommended_action
- key findings
- evidence citations
- clinician_message
- pharmacist_note
"""

try:
    llm_review = await run_async(review_prompt)
    rprint(llm_review)
except Exception as exc:
    print("LLM endpoint was not available or returned an error.")
    print("Using deterministic baseline review instead.")
    print(type(exc).__name__, str(exc)[:500])
    rprint(baseline_review)

You can also run a quick plain-English question against the same tools.

In [ ]:
question = "Should the pharmacist dispense TMP-SMX for patient-001 right now? Explain in 5 bullets with citations."

try:
    answer = await run_async(question)
    rprint(answer)
except Exception as exc:
    print("LLM endpoint unavailable. Baseline action:", baseline_review["recommended_action"])
    for finding in baseline_review["key_findings"][:5]:
        print("-", finding["severity"], finding["title"])

<a id="step9"></a>

## Step 9: Evaluating the solution

A competitive hackathon submission should show measurable value:

- **Clinical quality:** recall for critical findings, precision of actionable alerts, and gold-label agreement
- **Efficiency:** P50/P95 latency, tokens per review, and reviews per minute per GPU
- **Safety:** unsupported-claim rate, missing-data detection, citation coverage, and human-in-the-loop rate

In [ ]:
import time

GOLD = {
    "patient-001": {
        "required_titles": {
            "Sulfonamide antibiotic allergy conflict",
            "TMP-SMX may increase warfarin anticoagulation effect",
            "Supratherapeutic INR before interacting antibiotic",
        },
        "expected_action": "block",
    }
}


def evaluate_review(patient_id: str, review: dict) -> dict:
    titles = {finding["title"] for finding in review["key_findings"]}
    required = GOLD[patient_id]["required_titles"]
    found_required = titles.intersection(required)
    recall = len(found_required) / len(required)
    action_match = review["recommended_action"] == GOLD[patient_id]["expected_action"]
    citation_coverage = sum(1 for f in review["key_findings"] if f["evidence_ids"]) / max(1, len(review["key_findings"]))
    return {
        "required_finding_recall": round(recall, 3),
        "action_match": action_match,
        "citation_coverage": round(citation_coverage, 3),
        "overall_risk": review["overall_risk"],
        "recommended_action": review["recommended_action"],
    }


start = time.perf_counter()
review = run_deterministic_dur_review("patient-001", proposed_rx)
latency_ms = (time.perf_counter() - start) * 1000

metrics = evaluate_review("patient-001", review)
metrics["deterministic_latency_ms"] = round(latency_ms, 2)
metrics

## Enterprise architecture for the final pitch

For the hackathon presentation, position MedGuard AI as a FHIR-native, evidence-constrained review layer.

```text
EHR / Pharmacy System
     |
     v
FHIR Bundle + Proposed MedicationRequest
     |
     v
PydanticAI Orchestrator
     |
     +-- Medication Normalization Tool: RxNorm/RxNav
     +-- DUR Rules Tool: interactions, allergies, duplicate therapy
     +-- Patient Risk Tool: labs, age, renal/hepatic, pregnancy
     +-- Controlled Substance Tool: MME, opioid/benzo, PDMP stub
     +-- Evidence RAG Tool: DailyMed, labels, CDC guidance
     |
     v
Qwen3 on ROCm/vLLM
     |
     v
Structured ClinicalReviewDecision + Audit Trail
```

<a id="step10"></a>

## Step 10: Challenge ideas for the hackathon

Expand this prototype into a full enterprise-grade solution:

1. Replace the local drug map with live RxNorm/RxNav and DailyMed retrieval.
2. Generate 100 synthetic patients with Synthea and create gold-label test cases.
3. Add a vector database for labeling and guideline retrieval.
4. Add a UI with patient context, proposed prescription, findings, citations, and intervention note.
5. Add a controlled-substance dashboard with MME, opioid/benzodiazepine overlap, and PDMP simulation.
6. Add observability: latency, token usage, cache hits, severity distribution, and override reason.
7. Add a second Qwen3 "safety judge" agent that checks whether every clinical claim has evidence.

Final pitch line:

> MedGuard AI reduces alert fatigue by combining deterministic DUR screening, FHIR-native patient context, and evidence-grounded Qwen3 reasoning into a typed, auditable clinical review copilot.

# Enterprise Expansion

This section turns the workshop prototype into an enterprise-grade hackathon solution. It adds:

- Live RxNorm/RxNav and DailyMed retrieval with local fallback
- 100-patient synthetic cohort generation and gold labels
- Vector database retrieval for labels and guidelines
- Gradio UI for the clinical review workflow
- Controlled-substance dashboard with MME, opioid/benzodiazepine overlap, and PDMP simulation
- Observability for latency, tokens, cache hits, severity distribution, and override reasons
- A Qwen3 safety-judge agent plus deterministic fallback to verify evidence coverage

The cells are designed to run in two modes:

- **Demo mode:** offline deterministic review works with no external services.
- **Enterprise mode:** set `MEDGUARD_LIVE_APIS=1`, run vLLM/Qwen3, and install optional packages.

## Optional enterprise dependencies

These are optional. If install or network access is unavailable, the notebook falls back to standard-library retrieval and in-memory search.

In [ ]:
!pip install -q requests pandas numpy chromadb gradio scikit-learn

## 1. Live RxNorm/RxNav and DailyMed retrieval

Production systems should not rely on a hand-maintained medication dictionary. This cell wraps live NLM endpoints and falls back to the curated local map when the network is unavailable.

Official references:

- RxNav APIs expose RxNorm, RxTerms, Prescribable RxNorm, and RxClass resources.
- DailyMed REST services use `https://dailymed.nlm.nih.gov/dailymed/services/v2` and can return JSON or XML.

In [ ]:
import json
import os
import time
import urllib.parse
import urllib.request
from functools import lru_cache


RXNAV_BASE = "https://rxnav.nlm.nih.gov/REST"
DAILYMED_BASE = "https://dailymed.nlm.nih.gov/dailymed/services/v2"
LIVE_TERMINOLOGY_ENABLED = os.getenv("MEDGUARD_LIVE_APIS", "1") != "0"
LOCAL_NORMALIZE_MEDICATION_NAME = normalize_medication_name

DRUG_DB.update(
    {
        "amoxicillin": {
            "ingredients": ["amoxicillin"],
            "classes": ["penicillin_antibiotic", "beta_lactam_antibiotic"],
            "controlled": False,
        },
        "nitrofurantoin": {
            "ingredients": ["nitrofurantoin"],
            "classes": ["nitrofuran_antibiotic"],
            "controlled": False,
        },
    }
)

EVIDENCE_KB.update(
    {
        "dailyMed_amoxicillin": {
            "source": "DailyMed",
            "title": "Amoxicillin labeling",
            "url": "https://dailymed.nlm.nih.gov/dailymed/",
            "note": "FDA labeling source for amoxicillin allergy and warning review.",
        },
        "dailyMed_metformin": {
            "source": "DailyMed",
            "title": "Metformin labeling",
            "url": "https://dailymed.nlm.nih.gov/dailymed/",
            "note": "FDA labeling source for renal function warnings and contraindication review.",
        },
        "dailyMed_nitrofurantoin": {
            "source": "DailyMed",
            "title": "Nitrofurantoin labeling",
            "url": "https://dailymed.nlm.nih.gov/dailymed/",
            "note": "FDA labeling source for renal function and adverse reaction review.",
        },
    }
)

TERMINOLOGY_STATS = {
    "rxnav_cache_hits": 0,
    "rxnav_cache_misses": 0,
    "dailymed_cache_hits": 0,
    "dailymed_cache_misses": 0,
    "live_errors": [],
}


def _url_json(url: str, params: dict | None = None, timeout: int = 8) -> dict:
    if params:
        url = f"{url}?{urllib.parse.urlencode(params)}"
    try:
        request = urllib.request.Request(url, headers={"Accept": "application/json"})
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))
    except Exception as exc:
        TERMINOLOGY_STATS["live_errors"].append(f"{type(exc).__name__}: {str(exc)[:160]}")
        return {"_error": str(exc), "_url": url}


@lru_cache(maxsize=2048)
def rxnav_find_rxcui(name: str) -> dict:
    TERMINOLOGY_STATS["rxnav_cache_misses"] += 1
    return _url_json(f"{RXNAV_BASE}/rxcui.json", {"name": name, "search": 2})


@lru_cache(maxsize=2048)
def rxnav_get_properties(rxcui: str) -> dict:
    TERMINOLOGY_STATS["rxnav_cache_misses"] += 1
    return _url_json(f"{RXNAV_BASE}/rxcui/{rxcui}/properties.json")


@lru_cache(maxsize=2048)
def rxnav_get_related(rxcui: str) -> dict:
    TERMINOLOGY_STATS["rxnav_cache_misses"] += 1
    return _url_json(f"{RXNAV_BASE}/rxcui/{rxcui}/allrelated.json")


@lru_cache(maxsize=2048)
def rxnav_get_classes(rxcui: str) -> dict:
    TERMINOLOGY_STATS["rxnav_cache_misses"] += 1
    return _url_json(f"{RXNAV_BASE}/rxclass/class/byRxcui.json", {"rxcui": rxcui})


@lru_cache(maxsize=2048)
def dailymed_search_spls(drug_name: str) -> dict:
    TERMINOLOGY_STATS["dailymed_cache_misses"] += 1
    return _url_json(f"{DAILYMED_BASE}/spls.json", {"drug_name": drug_name})


@lru_cache(maxsize=2048)
def dailymed_get_spl(setid: str) -> dict:
    TERMINOLOGY_STATS["dailymed_cache_misses"] += 1
    return _url_json(f"{DAILYMED_BASE}/spls/{setid}.json")


def _first_rxcui(payload: dict) -> str | None:
    try:
        return payload["idGroup"]["rxnormId"][0]
    except Exception:
        return None


def _rxclass_names(payload: dict) -> list[str]:
    names = []
    for item in payload.get("rxclassDrugInfoList", {}).get("rxclassDrugInfo", []):
        cls = item.get("rxclassMinConceptItem", {})
        name = cls.get("className")
        if name:
            names.append(name.lower().replace(" ", "_"))
    return sorted(set(names))


def _related_ingredients(payload: dict) -> list[str]:
    ingredients = []
    for group in payload.get("allRelatedGroup", {}).get("conceptGroup", []):
        if group.get("tty") in {"IN", "PIN", "MIN"}:
            for prop in group.get("conceptProperties", []):
                if prop.get("name"):
                    ingredients.append(prop["name"].lower())
    return sorted(set(ingredients))


def live_medication_profile(name: str) -> dict:
    local = LOCAL_NORMALIZE_MEDICATION_NAME(name)
    if not LIVE_TERMINOLOGY_ENABLED:
        return {**local, "source": "local_fallback", "live_status": "disabled"}

    rxcui_payload = rxnav_find_rxcui(name)
    rxcui = _first_rxcui(rxcui_payload)
    if not rxcui:
        return {**local, "source": "local_fallback", "live_status": "rxnav_no_match"}

    properties = rxnav_get_properties(rxcui)
    related = rxnav_get_related(rxcui)
    classes = rxnav_get_classes(rxcui)
    live_ingredients = _related_ingredients(related)
    live_classes = _rxclass_names(classes)

    profile = {
        **local,
        "source": "rxnav_live_plus_local_safety_overlay",
        "live_status": "ok",
        "rxcui": rxcui,
        "rxnorm_name": properties.get("properties", {}).get("name"),
        "ingredients": sorted(set(local.get("ingredients", []) + live_ingredients)),
        "classes": sorted(set(local.get("classes", []) + live_classes)),
    }
    return profile


def dailymed_label_evidence(drug_name: str, limit: int = 3) -> list[dict]:
    payload = dailymed_search_spls(drug_name)
    records = payload.get("data", []) or payload.get("spls", []) or []
    evidence = []
    for record in records[:limit]:
        setid = record.get("setid") or record.get("setId") or record.get("spl_set_id")
        title = record.get("title") or record.get("published_date") or drug_name
        if setid:
            evidence.append(
                {
                    "source": "DailyMed",
                    "title": str(title),
                    "url": f"https://dailymed.nlm.nih.gov/dailymed/drugInfo.cfm?setid={setid}",
                    "note": f"DailyMed SPL label match for {drug_name}.",
                    "setid": setid,
                }
            )
    return evidence


def normalize_medication_name(name: str) -> dict:
    """Enterprise replacement: live RxNorm/RxNav normalization with safe local fallback."""
    return live_medication_profile(name)


print("Live terminology enabled:", LIVE_TERMINOLOGY_ENABLED)
print(live_medication_profile(proposed_rx.drug_name))
print(dailymed_label_evidence(proposed_rx.drug_name, limit=1))

## 2. Generate 100 synthetic patients and gold-label test cases

If you have Synthea available, generate FHIR R4/US Core output with:

```bash
java -jar synthea-with-dependencies.jar -s 21 -p 100 --exporter.fhir.use_us_core_ig true Massachusetts Boston
```

The code below creates a deterministic synthetic cohort inside the notebook so the demo remains portable even without the Synthea JAR.

In [ ]:
import random
import uuid
from pathlib import Path

WORK_DIR = Path("..").resolve() / "work" if Path.cwd().name == "outputs" else Path("work")
WORK_DIR.mkdir(exist_ok=True)

CASE_TEMPLATES = [
    {
        "name": "warfarin_tmp_smx",
        "conditions": ["Atrial fibrillation", "Chronic kidney disease stage 3b"],
        "medications": ["warfarin", "lisinopril", "spironolactone"],
        "allergies": [{"substance": "sulfonamide antibiotics", "reaction": "anaphylaxis", "severity": "severe"}],
        "labs": {"eGFR": 38, "potassium": 5.3, "INR": 3.2},
        "rx": ProposedPrescription(
            drug_name="sulfamethoxazole-trimethoprim DS",
            dose="800 mg / 160 mg",
            route="oral",
            frequency="twice daily",
            duration="7 days",
            indication="urinary tract infection",
        ),
        "gold_action": "block",
        "gold_findings": ["allergy", "warfarin_interaction", "inr_monitoring", "hyperkalemia"],
    },
    {
        "name": "opioid_benzo_sleep_apnea",
        "conditions": ["Chronic pain", "Obstructive sleep apnea", "Anxiety disorder"],
        "medications": ["oxycodone", "alprazolam"],
        "allergies": [],
        "labs": {"eGFR": 76, "potassium": 4.3, "INR": 1.0},
        "rx": ProposedPrescription(
            drug_name="oxycodone",
            dose="5 mg",
            route="oral",
            frequency="every 6 hours as needed",
            duration="5 days",
            indication="acute pain",
        ),
        "gold_action": "clarify",
        "gold_findings": ["opioid_benzo_overlap", "sleep_apnea_risk", "pdmp_review"],
    },
    {
        "name": "metformin_low_egfr",
        "conditions": ["Type 2 diabetes mellitus", "Chronic kidney disease stage 4"],
        "medications": ["metformin", "lisinopril"],
        "allergies": [],
        "labs": {"eGFR": 24, "potassium": 4.8, "INR": 1.0},
        "rx": ProposedPrescription(
            drug_name="metformin",
            dose="1000 mg",
            route="oral",
            frequency="twice daily",
            duration="90 days",
            indication="diabetes",
        ),
        "gold_action": "clarify",
        "gold_findings": ["renal_dose_review"],
    },
    {
        "name": "penicillin_allergy_amoxicillin",
        "conditions": ["Acute bacterial sinusitis"],
        "medications": [],
        "allergies": [{"substance": "penicillin", "reaction": "hives", "severity": "high"}],
        "labs": {"eGFR": 82, "potassium": 4.2, "INR": 1.0},
        "rx": ProposedPrescription(
            drug_name="amoxicillin",
            dose="875 mg",
            route="oral",
            frequency="twice daily",
            duration="7 days",
            indication="sinusitis",
        ),
        "gold_action": "clarify",
        "gold_findings": ["beta_lactam_allergy"],
    },
    {
        "name": "low_risk_nitrofurantoin",
        "conditions": ["Uncomplicated urinary tract infection"],
        "medications": ["atorvastatin"],
        "allergies": [],
        "labs": {"eGFR": 88, "potassium": 4.0, "INR": 1.0},
        "rx": ProposedPrescription(
            drug_name="nitrofurantoin",
            dose="100 mg",
            route="oral",
            frequency="twice daily",
            duration="5 days",
            indication="urinary tract infection",
        ),
        "gold_action": "approve",
        "gold_findings": [],
    },
]


def _med(name: str) -> dict:
    defaults = {
        "warfarin": ("5 mg", "oral", "daily"),
        "lisinopril": ("20 mg", "oral", "daily"),
        "spironolactone": ("25 mg", "oral", "daily"),
        "metformin": ("1000 mg", "oral", "twice daily"),
        "oxycodone": ("5 mg", "oral", "every 6 hours as needed"),
        "alprazolam": ("0.5 mg", "oral", "as needed"),
        "atorvastatin": ("20 mg", "oral", "daily"),
    }
    dose, route, frequency = defaults.get(name, ("standard dose", "oral", "daily"))
    return {"name": name, "dose": dose, "route": route, "frequency": frequency}


def generate_synthetic_cohort(n: int = 100, seed: int = 21) -> tuple[dict, list[dict]]:
    rng = random.Random(seed)
    patients = {}
    gold_labels = []

    for i in range(n):
        template = CASE_TEMPLATES[i % len(CASE_TEMPLATES)]
        patient_id = f"synthetic-{i+1:03d}"
        age = rng.randint(28, 88)
        patient = {
            "resourceType": "Bundle",
            "patient": {
                "id": patient_id,
                "name": f"Synthetic Patient {i+1:03d}",
                "age": age,
                "sex": rng.choice(["female", "male"]),
            },
            "conditions": template["conditions"],
            "allergies": template["allergies"],
            "medications": [_med(m) for m in template["medications"]],
            "observations": {
                "eGFR": {"value": template["labs"]["eGFR"], "unit": "mL/min/1.73m2"},
                "potassium": {"value": template["labs"]["potassium"], "unit": "mmol/L"},
                "INR": {"value": template["labs"]["INR"], "unit": "ratio"},
            },
        }
        patients[patient_id] = patient
        gold_labels.append(
            {
                "patient_id": patient_id,
                "case_type": template["name"],
                "proposed_rx": template["rx"].model_dump(),
                "expected_action": template["gold_action"],
                "expected_findings": template["gold_findings"],
            }
        )

    return patients, gold_labels


synthetic_patients, gold_labels_100 = generate_synthetic_cohort(100)
PATIENTS.update(synthetic_patients)

patients_path = WORK_DIR / "synthetic_patients_100.jsonl"
labels_path = WORK_DIR / "gold_labels_100.jsonl"
patients_path.write_text("\n".join(json.dumps(v) for v in synthetic_patients.values()), encoding="utf-8")
labels_path.write_text("\n".join(json.dumps(v) for v in gold_labels_100), encoding="utf-8")

print("Generated patients:", len(synthetic_patients))
print("Gold labels:", len(gold_labels_100))
print("Saved:", patients_path, labels_path)
gold_labels_100[:3]

## 3. Vector database for labels and guideline retrieval

Chroma is used when available. The fallback retriever is a tiny bag-of-words index that keeps the notebook runnable without extra services.

In [ ]:
import math
import re
from collections import Counter

GUIDELINE_DOCS = [
    {
        "id": "dailymed_tmp_smx_warning",
        "title": "TMP-SMX label warning summary",
        "source": "DailyMed",
        "text": "Sulfamethoxazole trimethoprim labeling includes serious hypersensitivity warnings and clinically important interaction review needs.",
    },
    {
        "id": "warfarin_antibiotic_monitoring",
        "title": "Warfarin antibiotic monitoring",
        "source": "Clinical pharmacy protocol",
        "text": "Warfarin patients receiving interacting antibiotics require bleeding-risk review, INR monitoring, and possible dose adjustment.",
    },
    {
        "id": "hyperkalemia_trimethoprim",
        "title": "Trimethoprim potassium monitoring",
        "source": "Clinical pharmacy protocol",
        "text": "Trimethoprim can worsen potassium risk, especially in CKD or with ACE inhibitors, ARBs, or potassium-sparing diuretics.",
    },
    {
        "id": "cdc_opioid_benzo",
        "title": "Opioid and benzodiazepine risk",
        "source": "CDC",
        "text": "Concurrent opioid and benzodiazepine exposure increases sedation and respiratory depression risk and should prompt careful review.",
    },
    {
        "id": "renal_dosing",
        "title": "Renal dosing protocol",
        "source": "Institutional renal dosing guideline",
        "text": "Reduced eGFR should trigger medication dose adjustment review and monitoring requirements for renally cleared drugs.",
    },
]


def _tokens(text: str) -> Counter:
    return Counter(re.findall(r"[a-z0-9]+", text.lower()))


class FallbackVectorStore:
    def __init__(self, docs: list[dict]):
        self.docs = docs
        self.vectors = [_tokens(d["text"] + " " + d["title"]) for d in docs]

    @staticmethod
    def _cosine(a: Counter, b: Counter) -> float:
        keys = set(a) | set(b)
        dot = sum(a[k] * b[k] for k in keys)
        mag_a = math.sqrt(sum(v * v for v in a.values()))
        mag_b = math.sqrt(sum(v * v for v in b.values()))
        return dot / (mag_a * mag_b) if mag_a and mag_b else 0.0

    def query(self, query_text: str, n_results: int = 3) -> list[dict]:
        query_vec = _tokens(query_text)
        scored = [
            {**doc, "score": round(self._cosine(query_vec, vector), 4)}
            for doc, vector in zip(self.docs, self.vectors)
        ]
        return sorted(scored, key=lambda row: row["score"], reverse=True)[:n_results]


try:
    import chromadb

    chroma_client = chromadb.Client()
    guideline_collection = chroma_client.get_or_create_collection(name="medguard_guidelines")
    guideline_collection.upsert(
        ids=[doc["id"] for doc in GUIDELINE_DOCS],
        documents=[doc["text"] for doc in GUIDELINE_DOCS],
        metadatas=[{"title": doc["title"], "source": doc["source"]} for doc in GUIDELINE_DOCS],
    )
    VECTOR_BACKEND = "chroma"
except Exception as exc:
    guideline_collection = FallbackVectorStore(GUIDELINE_DOCS)
    VECTOR_BACKEND = f"fallback_bow ({type(exc).__name__})"


def retrieve_guideline_context(query: str, n_results: int = 3) -> list[dict]:
    if VECTOR_BACKEND == "chroma":
        results = guideline_collection.query(query_texts=[query], n_results=n_results)
        docs = []
        for i, doc in enumerate(results.get("documents", [[]])[0]):
            metadata = results.get("metadatas", [[]])[0][i] or {}
            docs.append(
                {
                    "id": results.get("ids", [[]])[0][i],
                    "title": metadata.get("title", ""),
                    "source": metadata.get("source", ""),
                    "text": doc,
                    "distance": (results.get("distances") or [[None]])[0][i],
                }
            )
        return docs
    return guideline_collection.query(query, n_results=n_results)


print("Vector backend:", VECTOR_BACKEND)
retrieve_guideline_context("warfarin trimethoprim INR bleeding potassium CKD", 3)

## 4. Controlled-substance dashboard

This adds MME calculation, opioid/benzodiazepine overlap, sleep-apnea risk, and a simulated PDMP signal.

In [ ]:
OPIOID_MME_FACTORS = {
    "oxycodone": 1.5,
    "hydrocodone": 1.0,
    "morphine": 1.0,
    "hydromorphone": 4.0,
    "tramadol": 0.1,
}


def parse_mg(dose_text: str) -> float:
    match = re.search(r"(\d+(?:\.\d+)?)\s*mg", dose_text.lower())
    return float(match.group(1)) if match else 0.0


def doses_per_day(frequency: str) -> float:
    text = frequency.lower()
    if "every 6" in text:
        return 4
    if "every 8" in text:
        return 3
    if "twice" in text or "bid" in text:
        return 2
    if "three" in text or "tid" in text:
        return 3
    if "four" in text or "qid" in text:
        return 4
    if "daily" in text:
        return 1
    if "as needed" in text:
        return 1
    return 1


def medication_mme(med: dict) -> float:
    profile = LOCAL_NORMALIZE_MEDICATION_NAME(med["name"])
    opioid = next((i for i in profile.get("ingredients", []) if i in OPIOID_MME_FACTORS), None)
    if not opioid:
        return 0.0
    return parse_mg(med.get("dose", "")) * doses_per_day(med.get("frequency", "")) * OPIOID_MME_FACTORS[opioid]


def simulate_pdmp(patient: dict) -> dict:
    active_names = [m["name"].lower() for m in patient["medications"]]
    opioid_count = sum(any(opioid in name for opioid in OPIOID_MME_FACTORS) for name in active_names)
    benzo_count = sum("alprazolam" in name or "diazepam" in name or "lorazepam" in name for name in active_names)
    return {
        "pdmp_available": False,
        "simulated_prescribers_90d": 2 if opioid_count and benzo_count else 1,
        "simulated_pharmacies_90d": 2 if opioid_count and benzo_count else 1,
        "early_refill_signal": bool(opioid_count and benzo_count),
        "note": "PDMP is simulated for hackathon demo; production would integrate state PDMP where permitted.",
    }


def controlled_substance_dashboard(patient_id: str, proposed: ProposedPrescription | None = None) -> dict:
    patient = PATIENTS[patient_id]
    meds = list(patient["medications"])
    if proposed:
        meds.append(
            {
                "name": proposed.drug_name,
                "dose": proposed.dose,
                "route": proposed.route,
                "frequency": proposed.frequency,
            }
        )

    rows = []
    total_mme = 0.0
    has_opioid = False
    has_benzo = False
    for med in meds:
        profile = LOCAL_NORMALIZE_MEDICATION_NAME(med["name"])
        classes = set(profile.get("classes", []))
        mme = medication_mme(med)
        total_mme += mme
        has_opioid = has_opioid or "opioid_analgesic" in classes or mme > 0
        has_benzo = has_benzo or "benzodiazepine" in classes
        rows.append(
            {
                "medication": med["name"],
                "dose": med["dose"],
                "frequency": med["frequency"],
                "classes": ", ".join(profile.get("classes", [])),
                "daily_mme": round(mme, 1),
            }
        )

    risk_flags = []
    if has_opioid and has_benzo:
        risk_flags.append("opioid_benzodiazepine_overlap")
    if "Obstructive sleep apnea" in patient.get("conditions", []) and has_opioid:
        risk_flags.append("sleep_apnea_respiratory_risk")
    if total_mme >= 50:
        risk_flags.append("mme_50_or_higher")

    return {
        "patient_id": patient_id,
        "medications": rows,
        "total_daily_mme": round(total_mme, 1),
        "risk_flags": risk_flags,
        "pdmp": simulate_pdmp(patient),
    }


controlled_substance_dashboard("patient-001", proposed_rx)

## 5. Observability

This tracker captures the signals judges and enterprise buyers care about: latency, token estimate, cache status, severity distribution, and override reason.

In [ ]:
import statistics
import time
from collections import defaultdict


class ObservabilityTracker:
    def __init__(self):
        self.events = []

    def log_review(self, event: dict) -> None:
        self.events.append(event)

    def summary(self) -> dict:
        if not self.events:
            return {}
        latencies = [event["latency_ms"] for event in self.events]
        severities = defaultdict(int)
        actions = defaultdict(int)
        overrides = defaultdict(int)
        for event in self.events:
            severities[event["overall_risk"]] += 1
            actions[event["recommended_action"]] += 1
            if event.get("override_reason"):
                overrides[event["override_reason"]] += 1
        return {
            "reviews": len(self.events),
            "p50_latency_ms": round(statistics.median(latencies), 2),
            "p95_latency_ms": round(sorted(latencies)[int(0.95 * (len(latencies) - 1))], 2),
            "avg_estimated_tokens": round(statistics.mean(event["estimated_tokens"] for event in self.events), 1),
            "severity_distribution": dict(severities),
            "action_distribution": dict(actions),
            "override_reasons": dict(overrides),
            "terminology_stats": TERMINOLOGY_STATS,
            "vector_backend": VECTOR_BACKEND,
        }


OBS = ObservabilityTracker()


def estimated_tokens(obj: object) -> int:
    return max(1, len(json.dumps(obj, default=str).split()))


def supplemental_enterprise_findings(patient: dict, proposed: ProposedPrescription) -> list[Finding]:
    findings = []
    drug = proposed.drug_name.lower()
    egfr = patient.get("observations", {}).get("eGFR", {}).get("value")

    if "amoxicillin" in drug:
        for allergy in patient.get("allergies", []):
            if "penicillin" in allergy.get("substance", "").lower():
                findings.append(
                    Finding(
                        category="allergy",
                        severity="high",
                        title="Penicillin allergy conflict with amoxicillin",
                        rationale=(
                            f"Patient has {allergy.get('severity', 'documented')} penicillin allergy "
                            f"with reaction '{allergy.get('reaction', 'unknown')}'."
                        ),
                        recommendation="Clarify allergy history and select a non-penicillin alternative if the reaction is clinically significant.",
                        evidence_ids=["dailyMed_amoxicillin"],
                    )
                )

    if "metformin" in drug and egfr is not None and egfr < 30:
        findings.append(
            Finding(
                category="dose_adjustment",
                severity="high",
                title="Metformin renal contraindication review",
                rationale=f"Patient eGFR is {egfr} mL/min/1.73m2, which requires metformin safety review.",
                recommendation="Clarify therapy and renal dosing policy before approving metformin continuation or initiation.",
                evidence_ids=["dailyMed_metformin"],
            )
        )

    if "nitrofurantoin" in drug and egfr is not None and egfr < 30:
        findings.append(
            Finding(
                category="dose_adjustment",
                severity="high",
                title="Nitrofurantoin renal function review",
                rationale=f"Patient eGFR is {egfr} mL/min/1.73m2, requiring nitrofurantoin appropriateness review.",
                recommendation="Clarify renal appropriateness or select an alternative.",
                evidence_ids=["dailyMed_nitrofurantoin"],
            )
        )

    return findings


def merge_supplemental_findings(review: dict, supplemental: list[Finding]) -> dict:
    if not supplemental:
        return review

    existing_titles = {finding["title"] for finding in review["key_findings"]}
    for finding in supplemental:
        if finding.title not in existing_titles:
            review["key_findings"].append(finding.model_dump())

    review["key_findings"] = sorted(
        review["key_findings"],
        key=lambda f: _severity_rank(f["severity"]),
        reverse=True,
    )
    review["overall_risk"] = max(
        (f["severity"] for f in review["key_findings"]),
        key=_severity_rank,
        default="low",
    )
    review["recommended_action"] = _recommended_action(review["overall_risk"])

    evidence_ids = sorted({eid for f in review["key_findings"] for eid in f.get("evidence_ids", [])})
    review["evidence"] = [EVIDENCE_KB[eid] for eid in evidence_ids if eid in EVIDENCE_KB]
    return review


def enterprise_review(patient_id: str, proposed: ProposedPrescription, override_reason: str | None = None) -> dict:
    start = time.perf_counter()
    review = run_deterministic_dur_review(patient_id, proposed)
    review = merge_supplemental_findings(
        review,
        supplemental_enterprise_findings(PATIENTS[patient_id], proposed),
    )
    guideline_context = retrieve_guideline_context(" ".join(f["title"] for f in review["key_findings"]), 3)
    controlled = controlled_substance_dashboard(patient_id, proposed)
    review["guideline_context"] = guideline_context
    review["controlled_substance_dashboard"] = controlled
    review["terminology_profile"] = live_medication_profile(proposed.drug_name)

    latency_ms = (time.perf_counter() - start) * 1000
    event = {
        "patient_id": patient_id,
        "overall_risk": review["overall_risk"],
        "recommended_action": review["recommended_action"],
        "latency_ms": round(latency_ms, 2),
        "estimated_tokens": estimated_tokens(review),
        "finding_count": len(review["key_findings"]),
        "override_reason": override_reason,
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }
    review["observability_event"] = event
    OBS.log_review(event)
    return review


enterprise_baseline = enterprise_review("patient-001", proposed_rx)
print(enterprise_baseline["observability_event"])
OBS.summary()

## 6. Qwen3 safety judge agent

The primary agent drafts the review. The safety judge checks whether every clinical finding has evidence and whether the final message introduces unsupported clinical claims.

In [ ]:
class SafetyJudgeDecision(BaseModel):
    passed: bool
    unsupported_findings: list[str] = Field(default_factory=list)
    missing_citations: list[str] = Field(default_factory=list)
    notes: str


def deterministic_safety_judge(review: dict) -> SafetyJudgeDecision:
    evidence_sources = {item["source"] for item in review.get("evidence", [])}
    missing = []
    unsupported = []

    for finding in review.get("key_findings", []):
        if not finding.get("evidence_ids"):
            missing.append(finding.get("title", "untitled finding"))

    risky_phrases = ["guarantees", "proves causality", "safe to ignore", "no risk"]
    combined_text = (review.get("clinician_message", "") + " " + review.get("pharmacist_note", "")).lower()
    for phrase in risky_phrases:
        if phrase in combined_text:
            unsupported.append(phrase)

    has_findings = bool(review.get("key_findings"))
    passed = not missing and not unsupported and (bool(evidence_sources) or not has_findings)
    return SafetyJudgeDecision(
        passed=passed,
        unsupported_findings=unsupported,
        missing_citations=missing,
        notes="Deterministic evidence coverage check completed.",
    )


safety_judge_system_prompt = """
You are MedGuard Safety Judge. Check the clinical review for evidence discipline.
Pass only if every clinical finding has a cited evidence source and the final notes do not introduce unsupported claims.
Return a SafetyJudgeDecision.
"""

try:
    safety_judge_agent = Agent(
        model=agent_model,
        system_prompt=safety_judge_system_prompt,
        output_type=SafetyJudgeDecision,
    ) if Agent is not None and agent_model is not None else None
except TypeError:
    safety_judge_agent = Agent(
        model=agent_model,
        system_prompt=safety_judge_system_prompt,
    ) if Agent is not None and agent_model is not None else None


async def qwen3_safety_judge(review: dict):
    if safety_judge_agent is None:
        return deterministic_safety_judge(review)
    prompt = "Judge this clinical review for evidence coverage and unsupported claims:\n" + json.dumps(review, indent=2)
    result = await safety_judge_agent.run(prompt)
    return result.output


safety_result = deterministic_safety_judge(enterprise_baseline)
safety_result

## 7. UI for clinical review workflow

The Gradio UI gives judges a concrete product experience: patient context, proposed prescription, findings, citations, controlled-substance dashboard, and observability.

In [ ]:
def medguard_ui_review(patient_id: str, drug_name: str, dose: str, route: str, frequency: str, duration: str, indication: str, override_reason: str):
    rx = ProposedPrescription(
        drug_name=drug_name,
        dose=dose,
        route=route,
        frequency=frequency,
        duration=duration,
        indication=indication,
    )
    review = enterprise_review(patient_id, rx, override_reason=override_reason or None)
    judge = deterministic_safety_judge(review).model_dump()
    patient_context = PATIENTS[patient_id]

    findings_md = "\n".join(
        f"- **{f['severity'].upper()}** · {f['title']}: {f['recommendation']}"
        for f in review["key_findings"]
    ) or "- No major findings."

    citations_md = "\n".join(
        f"- [{e['source']}: {e['title']}]({e['url']})"
        for e in review.get("evidence", [])
    ) or "- No citations returned."

    summary_md = f"""
    ## Recommendation: `{review['recommended_action']}` · Risk: `{review['overall_risk']}`

    {review['clinician_message']}

    ### Findings
    {findings_md}

    ### Citations
    {citations_md}
    """
    return (
        json.dumps(patient_context, indent=2),
        summary_md,
        json.dumps(review["controlled_substance_dashboard"], indent=2),
        json.dumps(review["observability_event"], indent=2),
        json.dumps(judge, indent=2),
    )


try:
    import gradio as gr

    demo = gr.Interface(
        fn=medguard_ui_review,
        inputs=[
            gr.Dropdown(choices=sorted(PATIENTS.keys()), value="patient-001", label="Patient"),
            gr.Textbox(value=proposed_rx.drug_name, label="Drug"),
            gr.Textbox(value=proposed_rx.dose, label="Dose"),
            gr.Textbox(value=proposed_rx.route, label="Route"),
            gr.Textbox(value=proposed_rx.frequency, label="Frequency"),
            gr.Textbox(value=proposed_rx.duration, label="Duration"),
            gr.Textbox(value=proposed_rx.indication, label="Indication"),
            gr.Textbox(value="", label="Override reason, if clinician overrides"),
        ],
        outputs=[
            gr.Code(label="Patient Context"),
            gr.Markdown(label="Clinical Review"),
            gr.Code(label="Controlled Substance Dashboard"),
            gr.Code(label="Observability Event"),
            gr.Code(label="Safety Judge"),
        ],
        title="MedGuard AI Clinical Prescription Review Copilot",
        description="Synthetic data only. Clinical decision support prototype.",
    )
    print("Gradio UI created. Run demo.launch(share=True) in a notebook cell to open it.")
except Exception as exc:
    demo = None
    print("Gradio unavailable. UI function is still callable.")
    print(type(exc).__name__, str(exc)[:300])

medguard_ui_review("patient-001", proposed_rx.drug_name, proposed_rx.dose, proposed_rx.route, proposed_rx.frequency, proposed_rx.duration, proposed_rx.indication, "")[1]

## 8. Batch evaluation on the 100-patient gold-label set

The deterministic baseline is intentionally conservative. In the final submission, use this batch harness to compare:

- Rules-only baseline
- Qwen3 primary review agent
- Qwen3 primary review + safety judge
- Vector retrieval on/off

In [ ]:
def evaluate_gold_set(labels: list[dict], limit: int = 100) -> dict:
    rows = []
    for label in labels[:limit]:
        rx = ProposedPrescription(**label["proposed_rx"])
        review = enterprise_review(label["patient_id"], rx)
        judge = deterministic_safety_judge(review)
        rows.append(
            {
                "patient_id": label["patient_id"],
                "case_type": label["case_type"],
                "expected_action": label["expected_action"],
                "actual_action": review["recommended_action"],
                "action_match": review["recommended_action"] == label["expected_action"],
                "overall_risk": review["overall_risk"],
                "finding_count": len(review["key_findings"]),
                "safety_passed": judge.passed,
            }
        )
    return {
        "cases": len(rows),
        "action_accuracy": round(sum(row["action_match"] for row in rows) / max(1, len(rows)), 3),
        "safety_pass_rate": round(sum(row["safety_passed"] for row in rows) / max(1, len(rows)), 3),
        "sample_rows": rows[:10],
        "observability": OBS.summary(),
    }


batch_metrics = evaluate_gold_set(gold_labels_100, limit=100)
batch_metrics

## Production hardening checklist

Use this checklist in your final hackathon pitch:

- **FHIR integration:** consume `MedicationRequest`, `MedicationStatement`, `AllergyIntolerance`, `Condition`, `Observation`, and `Encounter`.
- **Terminology:** RxNorm/RxNav for normalization, RxClass for class membership, DailyMed for labeling.
- **Retrieval:** Chroma or enterprise vector DB for labels, institutional guidelines, renal dosing, opioid policy, and formulary.
- **Safety:** deterministic DUR first, Qwen3 second, safety judge last.
- **Governance:** audit trail, evidence ids, source URLs, model version, prompt version, override reason.
- **Performance:** cache RxNav/DailyMed calls, reuse embeddings, route easy reviews to rules-only mode, use Qwen3 thinking mode only for complex cases.
- **Human control:** block/clarify actions always require clinician or pharmacist decision.

## Sources Used for Enterprise Expansion

- RxNav APIs: https://lhncbc.nlm.nih.gov/RxNav/APIs/index.html
- DailyMed web services: https://dailymed.nlm.nih.gov/dailymed/app-support-web-services.cfm
- Synthea setup and running: https://github.com/synthetichealth/synthea/wiki/Basic-Setup-and-Running
- Chroma getting started: https://docs.trychroma.com/docs/overview/getting-started